[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/qMRData/blob/main/quantitative_imaging.ipynb)

In [ ]:
import importlib

if not importlib.util.find_spec("mrpro"):
    %pip install mrpro[notebooks]

## Quanitative Image Reconstruction

For the reconstruction of quantitative maps we will have to select the different signal models.

In [ ]:
import numpy as np
import torch
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData, IData, CsmData
from mrpro.data.traj_calculators import KTrajectoryCartesian
from mrpro.operators import DictionaryMatchOp
from mrpro.operators.models import MonoExponentialDecay, MOLLI
from cmap import Colormap

## Select the raw data files

In [ ]:
fname = "/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Raw/Left Knee/meas_MID00128_FID21426_t2map_trufi_Left.mrd"
fname_dicom = "/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Dicom/Left Knee/T2Map Trufi/t2map_trufi_Left_MOCO_T2_map"

# fname = '/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Raw/Left Knee/meas_MID00127_FID21425_t1map_MOLLI_Left.mrd'
# fname_dicom = '/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Dicom/Left Knee/T1Map MOLLI/t1map_MOLLI_Left_T1_map'

## Load Dicom

In [ ]:
qdata_dicom = IData.from_dicom_folder(fname_dicom)

## Reconstruct images

In [ ]:
kdata = KData.from_file(fname, trajectory=KTrajectoryCartesian())

# Resort the data into correct order of slices and contrasts
nslices = kdata.header.acq_info.idx.slice.unique().numel()
sort_idx = torch.argsort(kdata.header.acq_info.position.x.squeeze(), stable=True)
kdata = kdata[sort_idx].rearrange(
    "(slice contrast) ... -> contrast slice ...", slice=nslices
)

# We have to calculate the coil maps from one of the contrasts
csm = CsmData.from_kdata_inati(kdata[3], downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
idata = recon(kdata)

## Select signal model and calculate quantitative maps

We will select the correct signal model and then create a dictionary mapping operator, similarly what is used in MR Fingerprinting. 
We can then simply apply the dictionary mapping operator to the reconstructed images and obtain the quantitative maps

In [ ]:
if "t2map_trufi" in fname:
    t2_prep = [0.0, 0.025, 0.055, 0.0, 0.025, 0.06]
    dictionary = DictionaryMatchOp(
        MonoExponentialDecay(decay_time=t2_prep), index_of_scaling_parameter=0
    )
    dictionary.append(torch.tensor(1.0), torch.linspace(0.01, 0.8, 1000)[None, :])

    m0_match, qmap_match = dictionary(idata.data)
    qdata = IData(qmap_match, header=idata.header[0])
    cmap = Colormap("navia").to_mpl()
    vmax = 0.15
    colorbar_label = "T2 (s)"

elif "t1map_MOLLI" in fname:
    ti_times = 0.340 + torch.as_tensor([0, 1, 2, 3, 0, 1, 2, 0, 1, 0, 1])
    dictionary = DictionaryMatchOp(
        MOLLI(ti=torch.as_tensor(ti_times, dtype=torch.float32)),
        index_of_scaling_parameter=0,
    )
    dictionary.append(
        torch.tensor(1.0),
        torch.linspace(0.1, 3.0, 1000)[None, :],
        torch.linspace(0.1, 5.0, 1000)[None, None, :],
    )
    m0_match, c_match, qmap_match = dictionary(idata.data)
    qdata = IData(qmap_match, header=idata.header[0])
    cmap = Colormap("lipari").to_mpl()
    vmax = 1.2
    colorbar_label = "T1 (s)"

## Visualise results

In [ ]:
import matplotlib.pyplot as plt
from typing import Callable


def show_image(
    qmap: IData, cmap, vmax: float, colorbar_label: str, adapt_orientation: Callable
) -> None:
    """Show the qualitative images."""
    fig, ax = plt.subplots(1, 3, figsize=(8, 2))

    for cax in ax.flatten():
        cax.set_axis_off()

    def plot_multi_slice_image(ax, idata):
        """Plot three slices of M2D/3D image."""
        # Ensure the slices are in the correct order
        if idata.shape[0] > 1:
            sort_idx = torch.argsort(idata.header.position.x.squeeze())
            idata = idata[sort_idx]
        idim = idata.data.squeeze().shape[-3]
        for cax, slice_idx in zip(
            ax, [int(idim // 2 - idim // 4), idim // 2, int(idim // 2 + idim // 4)]
        ):
            im = cax.imshow(
                adapt_orientation(
                    idata.data.squeeze()[slice_idx].abs().numpy(force=True)
                ),
                cmap=cmap,
                vmin=0,
                vmax=vmax,
            )
            fig.colorbar(im, ax=cax, label=colorbar_label)

    plot_multi_slice_image(ax, qmap)

    plt.tight_layout()
    plt.show()

In [ ]:
show_image(qdata, cmap, vmax, colorbar_label, lambda x: np.rot90(x, 1)[:, ::-1])
show_image(qdata_dicom, cmap, vmax, colorbar_label, lambda x: np.rot90(x * 0.001, -1))